# 🏢 Project 8: Industry-Ready End-to-End ML System for Business Intelligence
### RISE Internship — Machine Learning & AI | Tamizan Skills

---

## 📌 Problem Statement
Many organizations experiment with machine learning models but fail to integrate them into end-to-end business workflows. Isolated models without insights, evaluation, and reporting do not create business value. Industry requires complete ML systems that support decision-making.

## 🎯 Objective
To design and implement an end-to-end machine learning solution that transforms raw business data into predictions, insights, and actionable intelligence.

## 🛠️ Tools
Python | NumPy | Pandas | Matplotlib | Seaborn | Scikit-learn | Jupyter Notebook

---

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               RandomForestRegressor, GradientBoostingRegressor)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
import os
os.makedirs('outputs', exist_ok=True)
print('✅ All libraries imported!')

---
## Step 2: Business Problem Definition
> This is what separates Project 8 from all others — we start with **business questions**, not just a dataset. The ML models serve the business, not the other way around.

In [ ]:
print("""
BUSINESS CONTEXT:
A retail company wants to use its customer data to answer 3 key questions:

PROBLEM A — CLASSIFICATION:
  Which customers are likely to churn? (Binary: Yes/No)

PROBLEM B — REGRESSION:
  What is each customer's predicted lifetime value (CLV)? ($ amount)

PROBLEM C — SEGMENTATION:
  Which customers are high-risk AND high-value? (Priority for retention)

Pipeline: Data → Cleaning → EDA → Features → Models
          → Evaluation → Comparison → Insights → Visualisation
""")

---
## Step 3: Data Collection & Preprocessing Pipeline
> We simulate a real retail customer dataset with 2000 customers, realistic missing values, and two targets — one for classification (Churn) and one for regression (CLV).

In [ ]:
np.random.seed(42)
N = 2000

age               = np.random.randint(18, 70, N)
gender            = np.random.choice(['Male','Female'], N)
region            = np.random.choice(['North','South','East','West'], N)
tenure_months     = np.random.randint(1, 72, N)
num_purchases     = np.random.poisson(12, N)
avg_order_value   = np.random.exponential(85, N) + 20
total_spent       = num_purchases * avg_order_value * np.random.uniform(0.8, 1.2, N)
days_since_purch  = np.random.exponential(30, N).astype(int)
num_returns       = np.random.poisson(1.5, N)
num_complaints    = np.random.poisson(0.8, N)
loyalty_score     = np.random.uniform(0, 100, N)
email_open_rate   = np.random.uniform(0, 1, N)
support_tickets   = np.random.poisson(1, N)
discount_used     = np.random.uniform(0, 0.4, N)

# CLV — regression target
clv = np.clip(
    total_spent * 0.3 + loyalty_score * 8 + tenure_months * 12
    - num_complaints * 150 - num_returns * 50
    + np.random.normal(0, 200, N), 50, None
)

# Churn — classification target
churn_prob = (
    0.3 * (days_since_purch > 60).astype(float)
    + 0.25 * (num_complaints > 2).astype(float)
    + 0.2  * (loyalty_score < 30).astype(float)
    + 0.15 * (tenure_months < 6).astype(float)
    - 0.2  * (email_open_rate > 0.5).astype(float)
    + np.random.normal(0, 0.1, N)
)
churn = (churn_prob > 0.35).astype(int)

df = pd.DataFrame({
    'CustomerID'        : [f'C{str(i).zfill(5)}' for i in range(1, N+1)],
    'Age'               : age, 'Gender': gender, 'Region': region,
    'TenureMonths'      : tenure_months, 'NumPurchases': num_purchases,
    'AvgOrderValue'     : avg_order_value.round(2),
    'TotalSpent'        : total_spent.round(2),
    'DaysSincePurchase' : days_since_purch,
    'NumReturns'        : num_returns, 'NumComplaints': num_complaints,
    'LoyaltyScore'      : loyalty_score.round(2),
    'EmailOpenRate'     : email_open_rate.round(4),
    'SupportTickets'    : support_tickets,
    'DiscountUsed'      : discount_used.round(4),
    'CLV'               : clv.round(2), 'Churn': churn,
})

# Inject realistic missing values
for col in ['AvgOrderValue','LoyaltyScore','EmailOpenRate']:
    mask = np.random.choice([True,False], N, p=[0.03,0.97])
    df.loc[mask, col] = np.nan

print(f'Shape: {df.shape}  |  Churn rate: {df["Churn"].mean()*100:.1f}%  |  Avg CLV: ${df["CLV"].mean():.2f}')
print(f'Missing values: {df.isnull().sum().sum()} cells')
df.head()

---
## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Business Intelligence — Customer Data Overview', fontsize=15, fontweight='bold')
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0,0])
churn_counts = df['Churn'].value_counts()
ax1.pie(churn_counts, labels=['Retained','Churned'], colors=['#2ecc71','#e74c3c'],
        autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
ax1.set_title('Churn Distribution')

ax2 = fig.add_subplot(gs[0,1])
ax2.hist(df['CLV'], bins=40, color='#3498db', alpha=0.8, edgecolor='black', linewidth=0.4)
ax2.axvline(df['CLV'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean=${df['CLV'].mean():.0f}")
ax2.set_title('CLV Distribution'); ax2.set_xlabel('CLV ($)'); ax2.legend(); ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(gs[0,2])
df[df['Churn']==0]['Age'].hist(bins=25, ax=ax3, alpha=0.6, color='#2ecc71', label='Retained', density=True)
df[df['Churn']==1]['Age'].hist(bins=25, ax=ax3, alpha=0.6, color='#e74c3c', label='Churned', density=True)
ax3.set_title('Age Distribution by Churn'); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(gs[1,0])
df.groupby('Region')['Churn'].mean().mul(100).plot(kind='bar', ax=ax4,
    color=['#3498db','#e74c3c','#2ecc71','#f39c12'], alpha=0.85, edgecolor='black', linewidth=0.5)
ax4.set_title('Churn Rate by Region (%)'); ax4.set_xticklabels(ax4.get_xticklabels(), rotation=0)
ax4.grid(axis='y', alpha=0.3)

ax5 = fig.add_subplot(gs[1,1])
sc = ax5.scatter(df['LoyaltyScore'], df['CLV'], c=df['Churn'], cmap='RdYlGn_r', alpha=0.4, s=10)
ax5.set_title('Loyalty Score vs CLV'); ax5.set_xlabel('Loyalty Score'); ax5.set_ylabel('CLV ($)')
plt.colorbar(sc, ax=ax5, label='Churn'); ax5.grid(alpha=0.3)

ax6 = fig.add_subplot(gs[1,2])
tenure_bins = pd.cut(df['TenureMonths'], bins=[0,6,12,24,36,72], labels=['0-6m','6-12m','1-2y','2-3y','3y+'])
df.groupby(tenure_bins, observed=True)['Churn'].mean().mul(100).plot(kind='bar', ax=ax6,
    color='#9b59b6', alpha=0.85, edgecolor='black', linewidth=0.5)
ax6.set_title('Churn Rate by Tenure'); ax6.set_xticklabels(ax6.get_xticklabels(), rotation=0)
ax6.grid(axis='y', alpha=0.3)

ax7 = fig.add_subplot(gs[2,0])
complaint_churn = df.groupby('NumComplaints')['Churn'].mean().mul(100)
ax7.bar(complaint_churn.index[:8], complaint_churn.values[:8], color='#e74c3c', alpha=0.85)
ax7.set_title('Churn Rate by # Complaints'); ax7.set_xlabel('Complaints'); ax7.grid(axis='y', alpha=0.3)

ax8 = fig.add_subplot(gs[2,1])
df.boxplot(column='CLV', by='Region', ax=ax8,
           medianprops=dict(color='red', linewidth=2))
plt.sca(ax8); plt.title('CLV by Region')

ax9 = fig.add_subplot(gs[2,2])
df[df['Churn']==0]['EmailOpenRate'].hist(bins=25, ax=ax9, alpha=0.6, color='#2ecc71', label='Retained', density=True)
df[df['Churn']==1]['EmailOpenRate'].hist(bins=25, ax=ax9, alpha=0.6, color='#e74c3c', label='Churned', density=True)
ax9.set_title('Email Open Rate by Churn'); ax9.legend(fontsize=8); ax9.grid(alpha=0.3)

plt.savefig('outputs/01_eda_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
numeric_cols = ['Age','TenureMonths','NumPurchases','AvgOrderValue','TotalSpent',
                'DaysSincePurchase','NumReturns','NumComplaints','LoyaltyScore',
                'EmailOpenRate','SupportTickets','DiscountUsed','CLV','Churn']

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(df[numeric_cols].corr(), dtype=bool))
sns.heatmap(df[numeric_cols].corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, linewidths=0.5, annot_kws={'size':7})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/02_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 5: Data Cleaning & Feature Engineering
> We handle missing values and engineer new features that capture business meaning — spend rate, complaint rate, engagement score — these derived features often outperform raw ones.

In [ ]:
df_clean = df.copy()

# Fill missing values with median
for col in ['AvgOrderValue','LoyaltyScore','EmailOpenRate']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Encode categoricals
le_gender = LabelEncoder(); df_clean['Gender_enc'] = le_gender.fit_transform(df_clean['Gender'])
le_region = LabelEncoder(); df_clean['Region_enc'] = le_region.fit_transform(df_clean['Region'])

# Engineered features — business meaning
df_clean['SpendPerMonth']   = df_clean['TotalSpent'] / (df_clean['TenureMonths'] + 1)
df_clean['ComplaintRate']   = df_clean['NumComplaints'] / (df_clean['NumPurchases'] + 1)
df_clean['ReturnRate']      = df_clean['NumReturns'] / (df_clean['NumPurchases'] + 1)
df_clean['EngagementScore'] = df_clean['EmailOpenRate'] * 50 + df_clean['LoyaltyScore'] * 0.5
df_clean['IsHighValue']     = (df_clean['CLV'] > df_clean['CLV'].quantile(0.75)).astype(int)
df_clean['IsRecentBuyer']   = (df_clean['DaysSincePurchase'] < 30).astype(int)
df_clean['IsLongTenure']    = (df_clean['TenureMonths'] > 24).astype(int)

feature_cols = [
    'Age','Gender_enc','Region_enc','TenureMonths',
    'NumPurchases','AvgOrderValue','TotalSpent','DaysSincePurchase',
    'NumReturns','NumComplaints','LoyaltyScore','EmailOpenRate',
    'SupportTickets','DiscountUsed',
    'SpendPerMonth','ComplaintRate','ReturnRate',
    'EngagementScore','IsHighValue','IsRecentBuyer','IsLongTenure'
]

print(f'Missing after cleaning : {df_clean[feature_cols].isnull().sum().sum()}')
print(f'Total features         : {len(feature_cols)}')
df_clean[feature_cols].head()

---
## Step 6: Train/Test Split & Feature Scaling

In [ ]:
X = df_clean[feature_cols]
y_cls = df_clean['Churn']
y_reg = df_clean['CLV']

X_train, X_test, y_cls_train, y_cls_test, y_reg_train, y_reg_test = train_test_split(
    X, y_cls, y_reg, test_size=0.2, random_state=42, stratify=y_cls
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {len(X_train)} customers  |  Test: {len(X_test)} customers')
print(f'Churn in test set: {y_cls_test.sum()} ({y_cls_test.mean()*100:.1f}%)')

---
## Step 7A: Problem A — Churn Classification
> Train 4 classifiers, compare them, pick the best.

In [ ]:
cls_models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cls_results = {}
for name, model in cls_models.items():
    model.fit(X_train_s, y_cls_train)
    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]
    cls_results[name] = {
        'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
        'accuracy' : accuracy_score(y_cls_test, y_pred),
        'precision': precision_score(y_cls_test, y_pred),
        'recall'   : recall_score(y_cls_test, y_pred),
        'f1'       : f1_score(y_cls_test, y_pred),
        'roc_auc'  : roc_auc_score(y_cls_test, y_prob),
    }
    print(f"  {name:22s}  F1={cls_results[name]['f1']:.4f}  AUC={cls_results[name]['roc_auc']:.4f}")

best_cls_name = max(cls_results, key=lambda k: cls_results[k]['f1'])
best_cls = cls_results[best_cls_name]
print(f'\n🏆 Best Classifier: {best_cls_name}')
print(classification_report(y_cls_test, best_cls['y_pred'], target_names=['Retained','Churned']))

---
## Step 7B: Problem B — CLV Regression
> Train 3 regressors to predict each customer's lifetime value in dollars.

In [ ]:
reg_models = {
    'Linear Regression' : LinearRegression(),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

reg_results = {}
for name, model in reg_models.items():
    model.fit(X_train_s, y_reg_train)
    y_pred = model.predict(X_test_s)
    reg_results[name] = {
        'model': model, 'y_pred': y_pred,
        'MAE' : mean_absolute_error(y_reg_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_reg_test, y_pred)),
        'R2'  : r2_score(y_reg_test, y_pred),
    }
    print(f"  {name:22s}  MAE=${reg_results[name]['MAE']:.2f}  R²={reg_results[name]['R2']:.4f}")

best_reg_name = max(reg_results, key=lambda k: reg_results[k]['R2'])
best_reg = reg_results[best_reg_name]
print(f'\n🏆 Best Regressor: {best_reg_name}  |  R²={best_reg["R2"]:.4f}')

---
## Step 8: Model Evaluation & Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Problem A — Churn Classification: Model Comparison', fontsize=13, fontweight='bold')

short = ['LR','DT','RF','GB']
x = np.arange(len(short)); w = 0.25
vals_f1  = [cls_results[n]['f1']       for n in cls_results]
vals_auc = [cls_results[n]['roc_auc']  for n in cls_results]
vals_acc = [cls_results[n]['accuracy'] for n in cls_results]

axes[0].bar(x-w, vals_f1,  w, label='F1',       color='#3498db', alpha=0.85)
axes[0].bar(x,   vals_auc, w, label='ROC-AUC',  color='#e74c3c', alpha=0.85)
axes[0].bar(x+w, vals_acc, w, label='Accuracy', color='#2ecc71', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(short)
axes[0].set_title('Classification Metrics'); axes[0].set_ylim(0, 1.1)
axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)

for (name, res), color in zip(cls_results.items(), ['#3498db','#e74c3c','#2ecc71','#f39c12']):
    fpr, tpr, _ = roc_curve(y_cls_test, res['y_prob'])
    axes[1].plot(fpr, tpr, label=f"{name[:12]} ({res['roc_auc']:.3f})", color=color, lw=2)
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_title('ROC Curves'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

cm = confusion_matrix(y_cls_test, best_cls['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'])
axes[2].set_title(f'Confusion Matrix — {best_cls_name}')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('outputs/03_classification_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Problem B — CLV Regression: Model Comparison', fontsize=13, fontweight='bold')
names_reg = list(reg_results.keys())

for ax, metric, color, title in zip(axes,
    ['MAE','RMSE','R2'], ['#3498db','#e74c3c','#2ecc71'],
    ['MAE — lower is better','RMSE — lower is better','R² — higher is better']):
    vals = [reg_results[n][metric] for n in names_reg]
    ax.bar(names_reg, vals, color=color, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.set_title(title); ax.set_xticklabels(names_reg, rotation=15, ha='right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/04_regression_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'CLV Prediction — {best_reg_name}', fontsize=13, fontweight='bold')

axes[0].scatter(y_reg_test, best_reg['y_pred'], alpha=0.3, color='#3498db', s=12)
mv, xv = min(y_reg_test.min(), best_reg['y_pred'].min()), max(y_reg_test.max(), best_reg['y_pred'].max())
axes[0].plot([mv,xv],[mv,xv],'r--', lw=2, label='Perfect')
axes[0].set_xlabel('Actual CLV ($)'); axes[0].set_ylabel('Predicted CLV ($)')
axes[0].set_title('Actual vs Predicted'); axes[0].legend(); axes[0].grid(alpha=0.3)

residuals_reg = y_reg_test.values - best_reg['y_pred']
axes[1].hist(residuals_reg, bins=40, color='#9b59b6', alpha=0.8, edgecolor='black', linewidth=0.4)
axes[1].axvline(0, color='red', lw=2, linestyle='--')
axes[1].set_title('Residuals Distribution'); axes[1].set_xlabel('Residual ($)'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/05_clv_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 9: Prediction & Insight Generation

In [ ]:
rf_cls = cls_results['Random Forest']['model']
rf_reg = reg_results['Random Forest']['model']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Feature Importance — Random Forest', fontsize=13, fontweight='bold')

pd.Series(rf_cls.feature_importances_, index=feature_cols).nlargest(12)\
  .sort_values().plot(kind='barh', ax=axes[0], color='#e74c3c', edgecolor='black', linewidth=0.5)
axes[0].set_title('Churn Classification'); axes[0].grid(axis='x', alpha=0.3)

pd.Series(rf_reg.feature_importances_, index=feature_cols).nlargest(12)\
  .sort_values().plot(kind='barh', ax=axes[1], color='#3498db', edgecolor='black', linewidth=0.5)
axes[1].set_title('CLV Regression'); axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/06_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10: Visualisation of Business Outcomes

In [ ]:
df_out = df_clean.copy()
X_all_s = scaler.transform(df_clean[feature_cols])
df_out['Churn_Probability'] = rf_cls.predict_proba(X_all_s)[:, 1]
df_out['Predicted_CLV']     = rf_reg.predict(X_all_s)
df_out['Churn_Risk']        = pd.cut(df_out['Churn_Probability'],
                                      bins=[0,0.3,0.6,1.0],
                                      labels=['Low Risk','Medium Risk','High Risk'])

print(f'High-Risk Customers   : {(df_out["Churn_Risk"]=="High Risk").sum()}')
print(f'Revenue at Risk       : ${df_out[df_out["Churn_Risk"]=="High Risk"]["Predicted_CLV"].sum():,.2f}')
df_out[['CustomerID','Churn_Probability','Predicted_CLV','Churn_Risk']].head(10)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Business Intelligence Dashboard — Predictions & Insights', fontsize=14, fontweight='bold')

risk_counts = df_out['Churn_Risk'].value_counts()
axes[0,0].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%',
              colors=['#2ecc71','#f39c12','#e74c3c'],
              startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0,0].set_title('Customers by Churn Risk Segment')

clv_by_risk = df_out.groupby('Churn_Risk', observed=True)['Predicted_CLV'].mean()
bars = axes[0,1].bar(clv_by_risk.index, clv_by_risk.values,
                     color=['#2ecc71','#f39c12','#e74c3c'], alpha=0.85, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, clv_by_risk.values):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                   f'${val:.0f}', ha='center', fontsize=9, fontweight='bold')
axes[0,1].set_title('Avg CLV by Risk Segment'); axes[0,1].set_ylabel('CLV ($)'); axes[0,1].grid(axis='y', alpha=0.3)

df_out[df_out['Churn']==0]['Churn_Probability'].hist(bins=30, ax=axes[0,2], alpha=0.6, color='#2ecc71', label='Retained', density=True)
df_out[df_out['Churn']==1]['Churn_Probability'].hist(bins=30, ax=axes[0,2], alpha=0.6, color='#e74c3c', label='Churned', density=True)
axes[0,2].axvline(0.5, color='black', linestyle='--', lw=1.5); axes[0,2].set_title('Churn Probability Distribution')
axes[0,2].legend(fontsize=8); axes[0,2].grid(alpha=0.3)

high_risk = df_out[df_out['Churn_Risk']=='High Risk'].nlargest(10, 'Predicted_CLV')
axes[1,0].barh(high_risk['CustomerID'], high_risk['Predicted_CLV'], color='#e74c3c', alpha=0.85)
axes[1,0].set_title('Top 10 High-Risk Customers\n(Priority Retention)'); axes[1,0].set_xlabel('CLV ($)'); axes[1,0].grid(axis='x', alpha=0.3)

df_out.boxplot(column='Predicted_CLV', by='Region', ax=axes[1,1], medianprops=dict(color='red', linewidth=2))
plt.sca(axes[1,1]); plt.title('CLV by Region')

rev_by_risk = df_out.groupby('Churn_Risk', observed=True)['Predicted_CLV'].sum()
axes[1,2].bar(rev_by_risk.index, rev_by_risk.values,
              color=['#2ecc71','#f39c12','#e74c3c'], alpha=0.85, edgecolor='black', linewidth=0.5)
axes[1,2].set_title('Total CLV at Stake by Risk Segment'); axes[1,2].set_ylabel('Total CLV ($)')
axes[1,2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}')); axes[1,2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/07_business_outcomes_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 11: Complete Project Documentation & Final Summary

In [ ]:
output_cols = ['CustomerID','Age','Region','TenureMonths','TotalSpent',
               'LoyaltyScore','NumComplaints','Churn',
               'Churn_Probability','Predicted_CLV','Churn_Risk']
df_out[output_cols].to_csv('outputs/customer_predictions.csv', index=False)
df_out[df_out['Churn_Risk']=='High Risk'][output_cols]\
    .sort_values('Predicted_CLV', ascending=False)\
    .to_csv('outputs/high_risk_customers.csv', index=False)

model_summary = pd.DataFrame({
    'Problem': ['Churn Classification']*4 + ['CLV Regression']*3,
    'Model'  : list(cls_results.keys()) + list(reg_results.keys()),
    'Metric1': [f"F1={v['f1']:.4f}"  for v in cls_results.values()] +
               [f"MAE={v['MAE']:.2f}" for v in reg_results.values()],
    'Metric2': [f"AUC={v['roc_auc']:.4f}" for v in cls_results.values()] +
               [f"R²={v['R2']:.4f}"  for v in reg_results.values()],
})
model_summary.to_csv('outputs/model_summary.csv', index=False)
print('✅ Saved: customer_predictions.csv')
print('✅ Saved: high_risk_customers.csv')
print('✅ Saved: model_summary.csv')

In [ ]:
rev_at_risk = df_out[df_out['Churn_Risk']=='High Risk']['Predicted_CLV'].sum()
high_risk_df = df_out[df_out['Churn_Risk']=='High Risk'].nlargest(5,'Predicted_CLV')

print('=' * 65)
print('   FINAL SUMMARY — END-TO-END ML SYSTEM')
print('=' * 65)
print(f'  PROBLEM A — CHURN CLASSIFICATION')
print(f'  Best Model : {best_cls_name}')
print(f'  F1-Score   : {best_cls["f1"]:.4f}')
print(f'  ROC-AUC    : {best_cls["roc_auc"]:.4f}')
print()
print(f'  PROBLEM B — CLV REGRESSION')
print(f'  Best Model : {best_reg_name}')
print(f'  MAE        : ${best_reg["MAE"]:.2f}')
print(f'  R²         : {best_reg["R2"]:.4f}')
print()
print(f'  BUSINESS IMPACT')
print(f'  High-Risk Customers : {(df_out["Churn_Risk"]=="High Risk").sum()}')
print(f'  Revenue at Risk     : ${rev_at_risk:,.2f}')
print(f'  Top 5 CLV at Risk   : ${high_risk_df["Predicted_CLV"].sum():,.2f}')
print('=' * 65)
print('   PROJECT 8 COMPLETE ✓')
print('=' * 65)

---
## 📊 Conclusion

### Problem A — Churn Classification
| Model | F1 | ROC-AUC |
|-------|----|---------|
| Gradient Boosting | 0.4783 | 0.9281 |
| Logistic Regression | 0.4091 | 0.8932 |
| Decision Tree | 0.4231 | 0.7617 |
| Random Forest | 0.3721 | 0.9258 |

### Problem B — CLV Regression
| Model | MAE | R² |
|-------|-----|----|
| Linear Regression | $147.85 | 0.8782 |
| Gradient Boosting | $160.44 | 0.8559 |
| Random Forest | $164.22 | 0.8536 |

### Key Business Takeaways
1. **110 customers** identified as High Risk — actionable retention list
2. **$82,678** in predicted CLV is at risk from churners
3. **Complaints, loyalty score, and days since purchase** are top churn predictors
4. **Total spent and loyalty** drive CLV the most
5. This pipeline goes from raw data to boardroom-ready insights in one notebook

---
*RISE Internship | Machine Learning & AI | Tamizan Skills | www.tamizhanskills.com*